# Transformation Validation & EDA

This notebook has two sections:

1. **Spec-conformance tests** — exercise every cleaning function in
   `src.data.transformations` against handcrafted inputs that cover every
   bullet in `docs/Data-Cleaning-Guide.md`.
2. **EDA on real processed data** — auto-load the highest-version
   `*_cleaned_v*_*.parquet` from `data/processed/` and inspect the cleaned schema,
   `valid_record` distribution, null rates, value frequencies, and a
   side-by-side raw→clean audit table.


In [ ]:
import os
import sys
import glob
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# Make `src` importable when the notebook runs from `notebooks/`
PROJECT_ROOT = Path(os.path.abspath('..'))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.transformations import (
    clean_first_name, clean_last_name, clean_middle_name, clean_suffix,
    clean_birth_date, clean_ssn,
    clean_address_line1, clean_address_line2, clean_city, clean_zip, clean_state,
    clean_phone, clean_email, clean_sex_at_birth,
    derive_full_name_tokens, derive_full_name_compact, derive_phones_set,
    transform_dataframe,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 60)

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'PROCESSED_DIR = {PROCESSED_DIR}')


---
## 1. Spec-conformance tests

A small harness compares actual vs. expected for each cleaner. Each section
covers the bullets in the corresponding guide section.

In [ ]:
def _values_equal(a, b):
    """NaN-/list-/Timestamp-aware equality."""
    if isinstance(a, tuple) and isinstance(b, tuple):
        return len(a) == len(b) and all(_values_equal(x, y) for x, y in zip(a, b))
    if isinstance(a, list) and isinstance(b, list):
        return a == b
    a_na = pd.isna(a) if not isinstance(a, (list, tuple)) else False
    b_na = pd.isna(b) if not isinstance(b, (list, tuple)) else False
    if a_na and b_na:
        return True
    if a_na or b_na:
        return False
    if isinstance(a, pd.Timestamp) or isinstance(b, pd.Timestamp):
        return pd.Timestamp(a) == pd.Timestamp(b)
    return a == b


def run_tests(label, fn, cases):
    """`cases` is a list of (input, expected, description) tuples. `input`
    may be a tuple of positional args (for multi-arg fns) or a single value."""
    print(f'=== {label} ===')
    n_pass = 0
    for raw_input, expected, desc in cases:
        args = raw_input if isinstance(raw_input, tuple) else (raw_input,)
        try:
            actual = fn(*args)
        except Exception as e:
            print(f'  [ERROR] {desc}: {type(e).__name__}: {e}')
            continue
        ok = _values_equal(actual, expected)
        n_pass += int(ok)
        status = 'PASS' if ok else 'FAIL'
        print(f'  [{status}] {desc}')
        if not ok:
            print(f'           input    = {raw_input!r}')
            print(f'           expected = {expected!r}')
            print(f'           actual   = {actual!r}')
    print(f'  → {n_pass}/{len(cases)} passed')
    print()


### 1.1 `FirstNM`

In [ ]:
run_tests('clean_first_name', clean_first_name, [
    ('WilliamEarl',        ('WILLIAM EARL', False), 'CamelCase split + uppercase'),
    ('MR JOSÉ',            ('JOSE', False),         'Unicode (É→E) + title strip'),
    ('  john   ',          ('JOHN', False),         'collapse whitespace + uppercase'),
    ('JOHN123',            ('JOHN', False),         'strip non-[A-Z space hyphen apostrophe]'),
    ("O'BRIEN",            ("O'BRIEN", False),      'apostrophe preserved'),
    ('ANNE-MARIE',         ('ANNE-MARIE', False),   'hyphen preserved'),
    ('JOHN BABYBOY',       ('JOHN BABYBOY', True),  'invalid contains: BABYBOY'),
    ('DUPLICATE ACCOUNT',  ('DUPLICATE ACCOUNT', True), 'invalid contains: DUPLICATE/ACCOUNT'),
    ('TEST',               ('TEST', True),          'invalid equals: TEST'),
    ('BABY',               ('BABY', True),          'invalid equals: BABY'),
    ('NULL',               (np.nan, False),         'text-null → NaN'),
    ('N/A',                (np.nan, False),         'N/A → NaN after char filter'),
    ('',                   (np.nan, False),         'empty string → NaN'),
    (np.nan,               (np.nan, False),         'NaN passthrough'),
    ('MRS SMITH',          ('SMITH', False),        'title MRS stripped'),
    ('DR ALEX',            ('ALEX', False),         'title DR stripped'),
])


### 1.2 `LastNM`

In [ ]:
run_tests('clean_last_name', clean_last_name, [
    ('RocaGarcia',         ('ROCA GARCIA', False),  'CamelCase split'),
    ('SMITH JR',           ('SMITH', False),        'generational suffix JR stripped'),
    ('SMITH III',          ('SMITH', False),        'III stripped (longest first)'),
    ('SMITH IV',           ('SMITH', False),        'IV stripped'),
    ('SMITH II',           ('SMITH', False),        'II stripped'),
    ('SMITH V',            ('SMITH', False),        'V stripped'),
    ('Ångström',           ('ANGSTROM', False),     'Unicode ligature/diacritic'),
    ('Иванов',             ('IVANOV', False),       'Cyrillic transliterated'),
    ("DON'T USE",          ("DON'T USE", True),     "invalid contains DON'T USE; apostrophe preserved by [A-Z \\-'] filter"),
    ('MEDICARE',           ('MEDICARE', True),      'invalid contains: MEDICARE'),
    ('UNKNOWN',            (np.nan, False),         'text-null → NaN'),
    (np.nan,               (np.nan, False),         'NaN passthrough'),
])


### 1.3 `MiddleNM`

In [ ]:
run_tests('clean_middle_name', clean_middle_name, [
    ('MARIE',              ('MARIE', False),        'simple value'),
    ('NMI',                (np.nan, False),         'middle-specific null NMI'),
    ('-',                  (np.nan, False),         'middle-specific null -'),
    ('UNKNOWN',            (np.nan, False),         'shared null UNKNOWN'),
    ('N/A',                (np.nan, False),         'shared null N/A'),
    ('BABYBOY',            ('BABYBOY', True),       'invalid contains'),
    ('TEST',               ('TEST', True),          'invalid equals: TEST'),
    ('',                   (np.nan, False),         'empty string → NaN'),
    (np.nan,               (np.nan, False),         'NaN passthrough'),
])


### 1.4 `SuffixNM`

In [ ]:
run_tests('clean_suffix', clean_suffix, [
    ('JR.',                'JR',                    'punctuation stripped'),
    ('Sr',                 'SR',                    'uppercase'),
    ('2ND',                'II',                    'ordinal 2ND→II'),
    ('3RD',                'III',                   'ordinal 3RD→III'),
    ('4TH',                'IV',                    'ordinal 4TH→IV'),
    ('5TH',                'V',                     'ordinal 5TH→V'),
    ('IV',                 'IV',                    'retain valid suffix'),
    ('XYZ',                np.nan,                  'invalid suffix → NaN'),
    ('UNKNOWN',            np.nan,                  'null → NaN'),
    (np.nan,               np.nan,                  'NaN passthrough'),
])


### 1.5 `BirthDT`

In [ ]:
ref = pd.Timestamp('2026-05-14')
run_tests('clean_birth_date', lambda v: clean_birth_date(v, reference_date=ref), [
    ('1985-06-15',         pd.Timestamp('1985-06-15'), 'standard ISO parse'),
    ('06/15/1985',         pd.Timestamp('1985-06-15'), 'US slash format'),
    ('1900-01-01',         pd.NaT,                     'cutoff: ≤1900-01-01 → NaT'),
    ('1850-01-01',         pd.NaT,                     'before 1900 → NaT'),
    ('2099-01-01',         pd.NaT,                     'future date → NaT'),
    ('not-a-date',         pd.NaT,                     'unparseable → NaT'),
    (np.nan,               pd.NaT,                     'NaN → NaT'),
])


### 1.6 `SSN`

In [ ]:
run_tests('clean_ssn', clean_ssn, [
    ('456-78-9012',        ('456789012', '9012'),   'valid SSN'),
    ('1234567',            ('001234567', '4567'),   '7-digit padded'),
    ('12345678',           ('012345678', '5678'),   '8-digit padded'),
    ('123-45-6789',        (np.nan, '6789'),        'sequential 123456789'),
    ('987654321',          (np.nan, '4321'),        'sequential descending'),
    ('111111111',          (np.nan, '1111'),        'all same digit'),
    ('123123123',          (np.nan, '3123'),        'repeating 3-digit block'),
    ('000-12-3456',        (np.nan, '3456'),        'area 000 invalid'),
    ('666-12-3456',        (np.nan, '3456'),        'area 666 invalid'),
    ('900-12-3456',        (np.nan, '3456'),        'area 9XX invalid'),
    ('111-00-3456',        (np.nan, '3456'),        'group 00 invalid'),
    ('111-22-0000',        (np.nan, '0000'),        'serial 0000 invalid'),
    ('010101010',          (np.nan, '1010'),        'known exact junk'),
    ('111223333',          (np.nan, '3333'),        'known exact junk: Woolworth'),
    ('',                   (np.nan, np.nan),        'empty string'),
    (np.nan,               (np.nan, np.nan),        'NaN passthrough'),
])


### 1.7 `AddressLine1`

In [ ]:
run_tests('clean_address_line1', clean_address_line1, [
    ('00123 N Main Street',  ('123 N MAIN ST', False),     'leading-zero strip + STREET→ST'),
    ('456 NORTHWEST PARKWAY',('456 NW PKWY', False),       'NORTHWEST→NW + PARKWAY→PKWY'),
    ('789 SOUTHEAST AVENUE', ('789 SE AVE', False),        'SOUTHEAST→SE + AVENUE→AVE'),
    ('123 W 45TH ST',        ('123 W 45TH ST', False),     'later digits preserved (no zero strip)'),
    ('100 STREETWISE LN',    ('100 STREETWISE LN', False), 'whole-word match: STREETWISE not changed'),
    ('22 BOULEVARD',         ('22 BLVD', False),           'BOULEVARD→BLVD'),
    ('5 SQUARE',             ('5 SQ', False),              'SQUARE→SQ'),
    ('HOMELESS',             (np.nan, False),              'placeholder → NaN (record stays valid)'),
    ('NO ADDRESS',           (np.nan, False),              'placeholder NO ADDRESS'),
    ('NULL',                 (np.nan, False),              'text-null NULL'),
    ('BABYBOY HOUSE',        ('BABYBOY HOUSE', True),      'invalid contains BABYBOY'),
    ('TEST',                 ('TEST', True),               'invalid equals TEST'),
    (np.nan,                 (np.nan, False),              'NaN passthrough'),
])


### 1.8 `AddressLine2`

In [ ]:
run_tests('clean_address_line2', clean_address_line2, [
    ('SUITE 100',            ('STE 100', False),           'SUITE→STE'),
    ('APARTMENT 5B',         ('APT 5B', False),            'APARTMENT→APT'),
    ('APRT 12',              ('APT 12', False),            'APRT→APT'),
    ('FLOOR 3',              ('FL 3', False),              'FLOOR→FL'),
    ('BUILDING 7',           ('BLDG 7', False),            'BUILDING→BLDG'),
    ('BLD 7',                ('BLDG 7', False),            'BLD→BLDG'),
    ('PENTHOUSE',            ('PH', False),                'PENTHOUSE→PH'),
    ('BASEMENT',             ('BSMT', False),              'BASEMENT→BSMT'),
    ('UNIT 4',               ('UNIT 4', False),            'UNIT unchanged (canonical)'),
    ('HOMELESS',             (np.nan, False),              'placeholder → NaN'),
    (np.nan,                 (np.nan, False),              'NaN passthrough'),
])


### 1.9 `CityNM`

In [ ]:
run_tests('clean_city', clean_city, [
    ('São Paulo',            'SAO PAULO',                  'Unicode'),
    ('Иваново',              'IVANOVO',                    'Cyrillic'),
    ('NEW YORK',             'NEW YORK',                   'multi-word preserved'),
    ("O'Fallon",             "O'FALLON",                   'apostrophe preserved'),
    ('Winston-Salem',        'WINSTON-SALEM',              'hyphen preserved'),
    ('Chicago1',             'CHICAGO',                    'digits stripped'),
    ('NULL',                 np.nan,                       'null text'),
    ('',                     np.nan,                       'empty string'),
    (np.nan,                 np.nan,                       'NaN passthrough'),
])


### 1.10 `ZipCD`

In [ ]:
run_tests('clean_zip', clean_zip, [
    ('60637',                ('60637', np.nan),            '5-digit base'),
    ('60637-1234',           ('60637', '1234'),            'ZIP+4 split'),
    ('606371234',            ('60637', '1234'),            'unhyphenated 9-digit split'),
    ('1234',                 ('01234', np.nan),            '4-digit padded to 5'),
    ('12345678',             ('01234', '5678'),            '8-digit padded to 9 and split'),
    ('00000',                (np.nan, np.nan),             'placeholder all-zeros'),
    ('11111',                (np.nan, np.nan),             'placeholder all-ones'),
    ('12345',                (np.nan, np.nan),             'sequential ascending'),
    ('54321',                (np.nan, np.nan),             'sequential descending'),
    ('123',                  (np.nan, np.nan),             'too short → NaN'),
    (np.nan,                 (np.nan, np.nan),             'NaN passthrough'),
])


### 1.11 `StateCD`

In [ ]:
run_tests('clean_state', clean_state, [
    ('CALIFORNIA',           'CA',                         'full name'),
    ('New York',             'NY',                         'full name mixed case'),
    ('TX',                   'TX',                         'valid USPS code passthrough'),
    ('05',                   'CA',                         'CCN numeric → USPS'),
    ('5',                    'CA',                         'CCN single-digit padded'),
    ('33',                   'NY',                         'CCN numeric → USPS'),
    ('XX',                   np.nan,                       'invalid → NaN'),
    ('NULL',                 np.nan,                       'null text'),
    (np.nan,                 np.nan,                       'NaN passthrough'),
])


### 1.12 Phone slots

In [ ]:
run_tests('clean_phone', clean_phone, [
    ('(312) 555-2000',       '3125552000',                 'parens stripped, digits 4-7=5552 (not in {5550,5551})'),
    ('1-312-555-2000',       '3125552000',                 'strip leading 1 (11-digit normalization)'),
    ('2125559876',           '2125559876',                 '555-9876: digits 4-7=5559, retained'),
    ('212-555-0100',         np.nan,                       'fictional range 555-0100..0199 (digits 4-7=5550)'),
    ('212-555-1100',         np.nan,                       'digits 4-7=5551 → guide rule nullifies'),
    ('1234567890',           np.nan,                       'sequential ascending'),
    ('9876543210',           np.nan,                       'sequential descending'),
    ('0000000000',           np.nan,                       'repeating all zeros'),
    ('5555555555',           np.nan,                       'repeating all fives'),
    ('911-555-1234',         np.nan,                       'N11 NPA 911'),
    ('555-123-4567',         np.nan,                       'unassigned NPA 555'),
    ('100-555-1234',         np.nan,                       'NPA starts with 1 → invalid'),
    ('212-100-1234',         np.nan,                       'NXX starts with 1 → invalid'),
    ('212-200-1234',         '2122001234',                 'valid'),
    (np.nan,                 np.nan,                       'NaN passthrough'),
])


### 1.13 `Email`

In [ ]:
run_tests('clean_email', clean_email, [
    ('VALID@DOMAIN.COM',     'valid@domain.com',           'uppercase normalized'),
    ('  user@domain.com  ',  'user@domain.com',            'whitespace stripped'),
    ('noreply@gmail.com',    np.nan,                       'local prefix noreply'),
    ('donotreply@anywhere.com', np.nan,                    'local prefix donotreply'),
    ('test@example.com',     np.nan,                       'exact junk + junk domain'),
    ('user123456@gmail.com', np.nan,                       'local contains 123456'),
    ('declined@hb.org',      np.nan,                       'contains: decline'),
    ('noemail@gmail.com',    np.nan,                       'contains: noemail'),
    ('ab@gmail.com',         np.nan,                       'local part length ≤ 2'),
    ('foo@example.org',      np.nan,                       'junk domain example.org'),
    ('foo@123.com',          np.nan,                       'junk domain 123.com'),
    ('missingat.com',        np.nan,                       'no @ → format regex fails'),
    ('a@b',                  np.nan,                       'no dot after @ → format regex fails'),
    ('foo@bar.com',          'foo@bar.com',                'valid'),
    (np.nan,                 np.nan,                       'NaN passthrough'),
])


### 1.14 `SexAtBirthDSC`

In [ ]:
run_tests('clean_sex_at_birth', clean_sex_at_birth, [
    ('MALE',                 'MALE',                       'retain MALE'),
    ('female',               'FEMALE',                     'lowercase normalized'),
    ('OTHER',                'OTHER',                      'retain OTHER'),
    ('unknown',              np.nan,                       'nullify UNKNOWN'),
    ('NULL',                 np.nan,                       'nullify NULL'),
    ('N/A',                  np.nan,                       'nullify N/A'),
    ('M',                    np.nan,                       'M alone not in guide retain-list'),
    ('F',                    np.nan,                       'F alone not in guide retain-list'),
    (np.nan,                 np.nan,                       'NaN passthrough'),
])


### 1.15 Cross-field derivations

In [ ]:
run_tests('derive_full_name_tokens', derive_full_name_tokens, [
    (('ANNE MARIE', None, "O'BRIEN-SMITH"), ['ANNE', 'MARIE', "O'BRIEN", 'SMITH'],
     'whitespace+hyphen split, apostrophe preserved, alphabetical'),
    (('JOHN', 'MICHAEL', 'SMITH'),           ['JOHN', 'MICHAEL', 'SMITH'],          'three-field union'),
    (('ANNE MARIE', None, 'ROCA GARCIA'),    ['ANNE', 'GARCIA', 'MARIE', 'ROCA'],   'multi-word in both fields'),
    ((np.nan, np.nan, np.nan),               [],                                    'all null → empty list'),
])

run_tests('derive_full_name_compact', derive_full_name_compact, [
    (('MARY', None, 'SMITH'),                'MARYSMITH',                            'standard concat'),
    (('MARY SMITH', None, None),             'MARYSMITH',                            'first-name carried last → same compact'),
    (('Anne-Marie', 'L', "O'Brien"),         'ANNEMARIELOBRIEN',                     'strip non-letters'),
    ((np.nan, np.nan, np.nan),               np.nan,                                 'all null → NaN'),
])

run_tests('derive_phones_set', derive_phones_set, [
    (('3125550150', '3125550150', None, '2125559876'),
        ['2125559876', '3125550150'],                                                'dedup + sort'),
    ((None, None, None, None),               [],                                     'all null → empty list'),
    (('6177772345',),                        ['6177772345'],                         'single phone'),
])


### 1.16 End-to-end `transform_dataframe`

Exercise the full orchestrator on a tiny synthetic frame to confirm column
renaming (`_raw`/`_clean`), `valid_record`, and the derived fields all wire up
together correctly.

In [ ]:
sample = pd.DataFrame({
    'PATID': ['P1', 'P2', 'P3', 'P4'],
    'FirstNM': ['WilliamEarl', 'MR JOSÉ', 'JOHN BABYBOY', None],
    'LastNM': ['RocaGarcia III', 'SMITH JR', 'DONOTUSE', None],
    'MiddleNM': ['NMI', 'MARIE', '', 'JANE'],
    'SuffixNM': ['JR.', '2ND', 'XYZ', None],
    'BirthDT': ['1985-06-15', '1850-01-01', '2099-01-01', 'not-a-date'],
    'SSN': ['123-45-6789', '456-78-9012', '666-00-1234', '0123456'],
    'AddressLine1': ['00123 N Main Street', '456 NORTHWEST PARKWAY', 'HOMELESS', 'BABYBOY HOUSE'],
    'AddressLine2': ['SUITE 100', 'APARTMENT 5B', None, None],
    'CityNM': ['São Paulo', 'Chicago', 'NULL', None],
    'ZipCD': ['02139', '60637', '00000', '90001'],
    'StateCD': ['Massachusetts', 'IL', 'XX', 'CA'],
    'CountryNM': ['USA', 'USA', None, 'USA'],
    'PrimaryPhoneNBR': ['(617) 555-2000', '1-312-555-2000', '0000000000', '6177772345'],
    'Phone01NBR': ['1-617-777-2345', None, '212-555-9876', None],
    'Phone02NBR': [None, None, None, None],
    'Phone03NBR': [None, None, None, None],
    'Email': ['valid@domain.com', 'noreply@gmail.com', 'test@example.com', 'a@b.c'],
    'SexAtBirthDSC': ['MALE', 'FEMALE', 'unknown', 'MALE'],
})

result = transform_dataframe(sample)

print('valid_record per row:')
print(result[['PATID', 'valid_record']].to_string(index=False))
print()

print('Name pipeline:')
print(result[['PATID', 'FirstNM_raw', 'FirstNM_clean', 'LastNM_raw', 'LastNM_clean',
              'MiddleNM_clean', 'SuffixNM_clean', 'full_name_tokens', 'full_name_compact']].to_string(index=False))
print()

print('Address pipeline:')
print(result[['PATID', 'AddressLine1_raw', 'AddressLine1_clean', 'AddressLine2_clean',
              'CityNM_clean', 'StateCD_clean', 'ZipCD_clean_base', 'ZipCD_clean_ext',
              'Address_normalized']].to_string(index=False))
print()

print('Other fields:')
print(result[['PATID', 'BirthDT_clean', 'SSN_clean', 'last_4_SSN', 'Email_clean',
              'SexAtBirthDSC_clean', 'Phones_set']].to_string(index=False))


---
## 2. EDA on the latest processed Parquet

Auto-discover the highest-version `*_cleaned_v*_*.parquet` file in
`data/processed/` and explore its cleaned schema, validity, null rates, value
distributions, and a raw→clean audit table.

In [ ]:
def find_latest_processed(processed_dir: Path):
    pattern = re.compile(r'^(?P<stem>.+)_cleaned_v(?P<n>\d+)(?:_\d{4}_\d{2}_\d{2})?\.parquet$')
    candidates = []
    if processed_dir.exists():
        for entry in processed_dir.iterdir():
            m = pattern.match(entry.name)
            if m:
                candidates.append((int(m.group('n')), entry))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]


latest = find_latest_processed(PROCESSED_DIR)
if latest is None:
    print(f'No *_cleaned_v*_*.parquet found in {PROCESSED_DIR}.')
    print('Drop a raw CSV in data/raw/ and run `python -m src.data.clean`, then re-run this notebook.')
    eda_df = None
else:
    print(f'Loading {latest.name} (modified {datetime.fromtimestamp(latest.stat().st_mtime):%Y-%m-%d %H:%M:%S})')
    eda_df = pd.read_parquet(latest)
    print(f'  rows:    {len(eda_df):,}')
    print(f'  columns: {len(eda_df.columns)}')


### 2.1 Column inventory

In [ ]:
if eda_df is not None:
    raw_cols = [c for c in eda_df.columns if c.endswith('_raw')]
    clean_cols = [c for c in eda_df.columns if c.endswith('_clean')]
    derived = [c for c in eda_df.columns if c in {
        'valid_record', 'last_4_SSN', 'ZipCD_clean_base', 'ZipCD_clean_ext',
        'full_name_tokens', 'full_name_compact', 'Phones_set', 'Address_normalized',
    }]
    other = [c for c in eda_df.columns if c not in set(raw_cols + clean_cols + derived)]

    print(f'_raw columns   ({len(raw_cols)}): {raw_cols}')
    print(f'_clean columns ({len(clean_cols)}): {clean_cols}')
    print(f'derived/global ({len(derived)}): {derived}')
    print(f'other          ({len(other)}): {other}')


### 2.2 `valid_record` distribution

In [ ]:
if eda_df is not None and 'valid_record' in eda_df.columns:
    valid = eda_df['valid_record'].astype(bool) if eda_df['valid_record'].dtype != bool else eda_df['valid_record']
    n_true = int(valid.sum())
    n_false = int((~valid).sum())
    total = len(eda_df)
    print(f'valid_record=True : {n_true:>8,} ({n_true/total:>6.1%})')
    print(f'valid_record=False: {n_false:>8,} ({n_false/total:>6.1%})')


### 2.3 Null rates per `_clean` column

In [ ]:
if eda_df is not None:
    clean_cols = sorted(c for c in eda_df.columns if c.endswith('_clean'))
    rows = []
    for col in clean_cols:
        n_null = int(eda_df[col].isna().sum())
        rows.append((col, n_null, n_null / len(eda_df)))
    nulls = pd.DataFrame(rows, columns=['column', 'n_null', 'null_rate']).sort_values('null_rate', ascending=False)
    display(nulls)


### 2.4 Top values per cleaned column

In [ ]:
if eda_df is not None:
    show_top = ['FirstNM_clean', 'LastNM_clean', 'CityNM_clean', 'StateCD_clean', 'SexAtBirthDSC_clean']
    for col in show_top:
        if col not in eda_df.columns:
            continue
        s = eda_df[col].dropna()
        if s.empty:
            continue
        print(f'--- {col}  ({len(s):,} non-null / {len(eda_df):,} total) ---')
        print(s.value_counts().head(10).to_string())
        print()


### 2.5 Sample of records marked invalid

In [ ]:
if eda_df is not None and 'valid_record' in eda_df.columns:
    invalid_mask = ~eda_df['valid_record'].astype(bool)
    n_invalid = int(invalid_mask.sum())
    print(f'{n_invalid:,} records marked invalid.')
    if n_invalid > 0:
        view_cols = [c for c in [
            'PATID', 'FirstNM_raw', 'FirstNM_clean', 'LastNM_raw', 'LastNM_clean',
            'AddressLine1_raw', 'AddressLine1_clean',
        ] if c in eda_df.columns]
        sample = eda_df.loc[invalid_mask, view_cols].head(15)
        display(sample)


### 2.6 Raw → clean audit (20 random rows)

In [ ]:
if eda_df is not None:
    audit_pairs = [
        ('FirstNM_raw', 'FirstNM_clean'),
        ('LastNM_raw', 'LastNM_clean'),
        ('AddressLine1_raw', 'AddressLine1_clean'),
        ('ZipCD_raw', 'ZipCD_clean_base'),
        ('StateCD_raw', 'StateCD_clean'),
        ('PrimaryPhoneNBR_raw', 'PrimaryPhoneNBR_clean'),
        ('Email_raw', 'Email_clean'),
    ]
    audit_cols = ['PATID'] if 'PATID' in eda_df.columns else []
    for raw, clean in audit_pairs:
        if raw in eda_df.columns and clean in eda_df.columns:
            audit_cols.extend([raw, clean])
    sample_n = min(20, len(eda_df))
    seed = 42 if len(eda_df) > sample_n else None
    audit = eda_df[audit_cols].sample(n=sample_n, random_state=seed) if seed else eda_df[audit_cols]
    display(audit)


### 2.7 Derived field samples

In [ ]:
if eda_df is not None:
    derived_cols = [c for c in [
        'PATID', 'full_name_tokens', 'full_name_compact', 'Phones_set', 'Address_normalized',
    ] if c in eda_df.columns]
    if len(derived_cols) > 1:
        display(eda_df[derived_cols].head(15))


### 2.8 Summary

In [ ]:
if eda_df is not None:
    print('=' * 60)
    print('Summary')
    print('=' * 60)
    print(f'File: {latest.name}')
    print(f'Total records: {len(eda_df):,}')
    if 'valid_record' in eda_df.columns:
        valid = eda_df['valid_record'].astype(bool)
        print(f'Valid records: {int(valid.sum()):,} ({valid.mean():.1%})')
    print(f'Generated:    {datetime.now():%Y-%m-%d %H:%M:%S}')
